# UCI BNN v2 — modular pipeline results

Loads the per-split `.pt` files written by `sazz.scripts.uci_bnn_module` and:
1. Computes posterior-predictive metrics from the saved samples (using `sazz.utils.metrics`)
2. Aggregates across splits if multiple are available

The runner now only persists samples + provenance; metrics are recomputed here so the
metrics module is the single source of truth for definitions.

Change `DATASET` and `SPLIT_ID` below to inspect a different run.

## Setup

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch

if Path.cwd().name == "notebooks":
    os.chdir("..")

from sazz.scripts.uci_bnn_module import (
    build_target, load_toy, load_raw_datasets, make_split,
    configs_for, BASE_SEED, TOY_DIR, UCI_DATASETS,
)
from sazz.utils.metrics import regression_metrics

torch.set_default_dtype(torch.float64)

# ---- Pick what to inspect ----
DATASET     = "boston"
SPLIT_ID    = 0
RESULTS_DIR = Path("results/uci_bnn_v2")

split_dir = RESULTS_DIR / DATASET / f"split_{SPLIT_ID:02d}"
print(f"Looking in {split_dir}")
print(f"  found: {sorted(p.name for p in split_dir.glob('*.pt'))}")

## Load all sampler runs for one (dataset, split)

In [ ]:
def load_runs(split_dir: Path) -> dict[str, dict]:
    return {p.stem: torch.load(p, weights_only=False)
            for p in sorted(split_dir.glob("*.pt"))}

runs = load_runs(split_dir)
print(f"Loaded {len(runs)} samplers: {list(runs)}")

if runs:
    example = next(iter(runs))
    print(f"\nKeys in {example}.pt: {list(runs[example])}")
    print(f"samples shape: {tuple(runs[example]['samples'].shape)}")
    print(f"pipeline: {runs[example].get('pipeline', '?')}")

## Rebuild the target so we can call `predict` on samples

Saved payloads carry the betas, not the predictions. To compute predictive
metrics we need a `target` so we can do `target.meta['model'].likelihood.predict(beta, X)`.
Rebuilding the target re-runs Adam to find `x_ref`; this is fast for the
architectures used here.

We also pull the `(X_test, y_test)` for this split from the same source the
runner used, so the test set we score against is exactly the one the chain
didn't see.

In [ ]:
def rebuild_for_split(payload: dict, dataset: str, split_id: int):
    """Reconstruct (data, cfg, target) for a saved run.

    Works for both toy 1D datasets and UCI datasets. Pulls the architecture
    fields out of the payload so the rebuilt target matches the one the
    chain was sampled from, even if the runner's defaults have changed.
    """
    if (TOY_DIR / f"{dataset}.pt").exists():
        data, cfg = load_toy(dataset, TOY_DIR)
    elif dataset in UCI_DATASETS:
        raw = load_raw_datasets()
        X, y = raw[dataset]
        data = make_split(X, y, seed=BASE_SEED + split_id)
        cfgs = configs_for({dataset: X.shape[1]})
        cfg = cfgs[dataset]
    else:
        raise ValueError(f"Unknown dataset: {dataset}")

    # Prefer payload values for architecture fields — these define what the
    # chain was actually sampled against.
    cfg.layer_sizes = payload["layer_sizes"]
    cfg.activation  = payload["activation"]
    cfg.noise_std   = payload["noise_std"]

    target = build_target(data, cfg)
    return data, cfg, target

first_payload = next(iter(runs.values()))
data, cfg, target = rebuild_for_split(first_payload, DATASET, SPLIT_ID)
print(f"target D = {target.D},  layer_sizes = {cfg.layer_sizes},  activation = {cfg.activation}")
print(f"X_test: {tuple(data['X_test'].shape)},  y_test: {tuple(data['y_test'].shape)}")

## Compute metrics from samples

`predict_regression` lives in `bnn_torch.py` — for a v2 target it goes through
`functional_call` on the saved module.

In [ ]:
@torch.no_grad()
def predict_from_samples(samples: torch.Tensor, target, X_test: torch.Tensor):
    """Posterior-predictive mean and std for a regression target.

    Works for any target whose likelihood exposes a .predict(beta, X) method,
    which is true for both v1 and v2 BNN likelihoods and for the GLM
    likelihoods.
    """
    likelihood = target.meta["model"].likelihood
    X_test = X_test.to(dtype=likelihood.X.dtype, device=likelihood.X.device)
    preds = torch.stack([
        likelihood.predict(beta, X_test).squeeze(-1) for beta in samples
    ])  # [n_samples, n_test]
    return preds.mean(0), preds.std(0)

rows = []
for name, payload in runs.items():
    samples = payload["samples"]
    mean_pred, std_pred = predict_from_samples(samples, target, data["X_test"])
    m = regression_metrics(
        y_true=data["y_test"], mean_pred=mean_pred, pred_std=std_pred,
        noise_std=payload["noise_std"], y_std=payload["y_std"],
        samples=samples,
    )
    rows.append({
        "sampler":     name,
        "n_samples":   samples.shape[0],
        "n_events":    payload.get("n_events"),
        "elapsed_sec": payload.get("elapsed_sec"),
        **m,
    })

df = pd.DataFrame(rows).set_index("sampler")
df.round(4)

## Aggregate across splits

Walks every `split_NN/*.pt` under `results/uci_bnn_v2/<dataset>/`, recomputes
metrics on each, and reports mean ± std-error across splits per (dataset, sampler).

Skip this cell if you only have a single split.

In [ ]:
def evaluate_split(dataset: str, split_id: int) -> list[dict]:
    """Compute metrics for every sampler in one (dataset, split).
    Returns a list of row dicts; empty if the split dir doesn't exist.
    """
    sd = RESULTS_DIR / dataset / f"split_{split_id:02d}"
    if not sd.exists():
        return []
    runs_ = load_runs(sd)
    if not runs_:
        return []

    first = next(iter(runs_.values()))
    data_, cfg_, target_ = rebuild_for_split(first, dataset, split_id)

    out = []
    for name, payload in runs_.items():
        samples = payload["samples"]
        mean_pred, std_pred = predict_from_samples(samples, target_, data_["X_test"])
        m = regression_metrics(
            y_true=data_["y_test"], mean_pred=mean_pred, pred_std=std_pred,
            noise_std=payload["noise_std"], y_std=payload["y_std"],
            samples=samples,
        )
        out.append({
            "dataset":     dataset,
            "split_id":    split_id,
            "sampler":     name,
            "elapsed_sec": payload.get("elapsed_sec"),
            **m,
        })
    return out

# Walk all dataset dirs that exist on disk
all_rows = []
for ds_dir in sorted(RESULTS_DIR.iterdir()):
    if not ds_dir.is_dir():
        continue
    for split_dir in sorted(ds_dir.glob("split_*")):
        sid = int(split_dir.name.split("_")[-1])
        all_rows.extend(evaluate_split(ds_dir.name, sid))

all_df = pd.DataFrame(all_rows)
print(f"Loaded {len(all_df)} (dataset, split, sampler) rows.")
all_df.head()

In [ ]:
if not all_df.empty:
    # ESS-per-second diagnostics
    all_df["ess_min_per_sec"]    = all_df["ess_min"]    / all_df["elapsed_sec"]
    all_df["ess_median_per_sec"] = all_df["ess_median"] / all_df["elapsed_sec"]

    summary = (
        all_df.groupby(["dataset", "sampler"])
              .agg(
                  rmse_orig_mean=("rmse_orig", "mean"),
                  rmse_orig_sem=("rmse_orig", "sem"),
                  log_lik_mean=("log_lik", "mean"),
                  log_lik_sem=("log_lik", "sem"),
                  ess_min_per_sec=("ess_min_per_sec", "mean"),
                  ess_median_per_sec=("ess_median_per_sec", "mean"),
                  elapsed_sec=("elapsed_sec", "mean"),
              )
              .round(4)
    )
    summary

In [ ]:
summary

In [ ]:
# Optional: dump the per-split metrics back to disk so you can compare
# against v1 metrics later.
# if not all_df.empty:
#     out_path = RESULTS_DIR / "metrics.csv"
#     all_df.to_csv(out_path, index=False)
#     print(f"Wrote {out_path}  ({len(all_df)} rows)")